# Unit 3 — Agentic loops ⭐

The loop is automatic. Our job is to make it visible.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from collections.abc import Awaitable, Callable
from typing import Annotated
from pydantic import Field
from agent_framework import FunctionInvocationContext, function_middleware
from agent_framework.openai import OpenAIChatCompletionClient

## Four tools

Enough that the agent has to plan an order.

In [ ]:
def get_weather(city: Annotated[str, Field(description="City name.")],
                date: Annotated[str, Field(description="Date like 'today' or 'tomorrow'.")]) -> str:
    """Get overall weather (temperature and conditions) for a city on a date."""
    return f"{city} on {date}: 18°C, partly sunny"

def get_forecast_hourly(city: Annotated[str, Field(description="City name.")]) -> str:
    """Get the hourly forecast for today and tomorrow."""
    return f"{city}: morning light rain, clear from noon onward, dry evening"

def find_parks(city: Annotated[str, Field(description="City name.")]) -> str:
    """List parks in a city suitable for a group picnic."""
    return f"{city} parks: Stadtpark (central), Prater (large, popular), Augarten (quieter)"

def suggest_food(weather: Annotated[str, Field(description="Weather description.")]) -> str:
    """Suggest picnic food suitable for given weather conditions."""
    return "light salads, fresh fruit, cold drinks, crusty bread"

## Middleware: log every tool call

The `@function_middleware` decorator marks this as a **function-invocation** middleware — it fires around each tool call. `context.function.name` and `context.arguments` before; `context.result` after `call_next()`.

In [ ]:
def _args_to_dict(args):
    if args is None:
        return {}
    if hasattr(args, "model_dump"):
        return args.model_dump()
    try:
        return dict(args)
    except (TypeError, ValueError):
        return {"_repr": str(args)}

@function_middleware
async def log_tool_calls(context: FunctionInvocationContext, call_next: Callable[[], Awaitable[None]]) -> None:
    name = context.function.name
    args = _args_to_dict(context.arguments)
    print(f"  [→] {name}({args})")
    await call_next()
    result_str = str(context.result)[:100]
    print(f"  [←] {name} → {result_str}")

## Run a multi-step task

Watch the tool calls scroll past before the final answer.

In [ ]:
agent = OpenAIChatCompletionClient().as_agent(
    name="PicnicPlanner",
    instructions="You help plan picnics. Use the tools to gather weather, forecast, parks, and food before giving a final plan. Keep the plan to 3-4 sentences.",
    tools=[get_weather, get_forecast_hourly, find_parks, suggest_food],
    middleware=[log_tool_calls],
)

result = await agent.run("Plan a picnic for tomorrow in Vienna. Tell me the time of day, where to go, and what to bring.")
print("\n---\n", result.text)

## Tinker

Ask the same question but with no middleware attached, and see how little you learn.

In [ ]:
silent_agent = OpenAIChatCompletionClient().as_agent(
    name="SilentPlanner",
    instructions="You help plan picnics. Use the tools.",
    tools=[get_weather, get_forecast_hourly, find_parks, suggest_food],
)
result = await silent_agent.run("Plan a picnic for tomorrow in Vienna.")
print(result.text)